In [1]:
import pandas as pd
import os

file_path = "../data/raw/road-accidents-annual-2020-2024.csv"

df_accident = pd.read_csv(file_path)

print("Dataset shape:", df_accident.shape)
print("\nColumn names:")
print(df_accident.columns.tolist())

display(df_accident.head())

Dataset shape: (5, 7)

Column names:
['Year', 'Accidents', '% Change from previous year', 'Fatalities', '% Change from previous year.1', 'Persons Injured', '% Change from previous year.2']


,Year,Accidents,% Change from previous year,Fatalities,% Change from previous year.1,Persons Injured,% Change from previous year.2
0,2020,"3,72,181",-18.6,"1,38,383",-13.0,"3,46,747",-22.8
1,2021,"4,12,432",10.8,"1,53,972",11.3,"3,84,448",10.9
2,2022,"4,61,312",11.9,"1,68,491",9.4,"4,43,366",15.3
3,2023,"4,80,583",4.2,"1,72,890",2.6,"4,62,825",4.4
4,2024,"4,87,707",1.5,"1,77,175",2.5,"4,71,441",1.9


In [2]:

# Copy the original data
df_accident_clean = df_accident.copy()

# Convert Indian comma-formatted numbers into numeric values
number_cols = ["Accidents", "Fatalities", "Persons Injured"]

for col in number_cols:
    df_accident_clean[col] = (
        df_accident_clean[col]
        .astype(str)
        .str.replace(",", "", regex=False)
        .astype(int)
    )

# Calculate accident severity indicators
df_accident_clean["fatality_rate_per_100_accidents"] = (
    df_accident_clean["Fatalities"]
    / df_accident_clean["Accidents"] * 100
).round(2)

df_accident_clean["injury_rate_per_100_accidents"] = (
    df_accident_clean["Persons Injured"]
    / df_accident_clean["Accidents"] * 100
).round(2)

display(df_accident_clean)


,Year,Accidents,% Change from previous year,Fatalities,% Change from previous year.1,Persons Injured,% Change from previous year.2,fatality_rate_per_100_accidents,injury_rate_per_100_accidents
0,2020,372181,-18.6,138383,-13.0,346747,-22.8,37.18,93.17
1,2021,412432,10.8,153972,11.3,384448,10.9,37.33,93.21
2,2022,461312,11.9,168491,9.4,443366,15.3,36.52,96.11
3,2023,480583,4.2,172890,2.6,462825,4.4,35.98,96.30
4,2024,487707,1.5,177175,2.5,471441,1.9,36.33,96.66


In [3]:
import os

os.makedirs("../data/processed", exist_ok=True)

df_accident_clean.to_csv(
    "../data/processed/india_accident_trends_cleaned.csv",
    index=False
)

print("Cleaned accident data saved successfully!")

Cleaned accident data saved successfully!


In [4]:

import os
import shutil

os.makedirs("../data/final", exist_ok=True)

shutil.copy2(
    "../data/processed/india_accident_trends_cleaned.csv",
    "../data/final/india_accident_trends_cleaned.csv"
)

print("Accident dataset saved in data/final!")


Accident dataset saved in data/final!


In [6]:

import pandas as pd

file_path = "../data/raw/ADSI_Table_1A.2_2.csv"

df_city_accidents = pd.read_csv(file_path, encoding="utf-8-sig")

print("Dataset shape:", df_city_accidents.shape)
print("\nColumn names:")
print(df_city_accidents.columns.tolist())

display(df_city_accidents.head(10))


Dataset shape: (93, 14)

Column names:
['Sl. No.', 'State/UT/City', 'Road Accidents - Cases', 'Road Accidents - Injured', 'Road Accidents - Died', 'Railway Accidents - Cases', 'Railway Accidents - Injured', 'Railway Accidents - Died', 'Railway Crossing Accidents - Cases', 'Railway Crossing Accidents - Injured', 'Railway Crossing Accidents - Died', 'Total Traffic Accidents - Cases', 'Total Traffic Accidents - Injured', 'Total Traffic Accidents - Died']


,Sl. No.,State/UT/City,Road Accidents - Cases,Road Accidents - Injured,Road Accidents - Died,Railway Accidents - Cases,Railway Accidents - Injured,Railway Accidents - Died,Railway Crossing Accidents - Cases,Railway Crossing Accidents - Injured,Railway Crossing Accidents - Died,Total Traffic Accidents - Cases,Total Traffic Accidents - Injured,Total Traffic Accidents - Died
0,1,Andhra Pradesh,21070,21340,8293,1029,0,1037,0,0,0,22099,21340,9330
1,2,Arunachal Pradesh,215,177,152,0,0,0,0,0,0,215,177,152
2,3,Assam,7028,5679,3060,689,0,693,22,0,22,7739,5679,3775
3,4,Bihar,10801,7068,8898,1444,6,1446,52,0,53,12297,7074,10397
4,5,Chhattisgarh,13091,11459,5890,429,21,429,4,0,4,13524,11480,6323
5,6,Goa,3012,1071,274,54,0,54,0,0,0,3066,1071,328
6,7,Gujarat,15777,15139,7634,766,0,777,6,0,6,16549,15139,8417
7,8,Haryana,10654,8353,5228,1127,26,1101,95,0,95,11876,8379,6424
8,9,Himachal Pradesh,2484,3891,979,0,0,0,0,0,0,2484,3891,979
9,10,Jharkhand,5175,3747,3898,362,19,346,7,0,7,5544,3766,4251


In [7]:

# Keep only road accident data
df_accident_score = df_city_accidents[
    ["State/UT/City", "Road Accidents - Cases",
     "Road Accidents - Died", "Road Accidents - Injured"]
].copy()

# Convert accident columns to numeric
cols = [
    "Road Accidents - Cases",
    "Road Accidents - Died",
    "Road Accidents - Injured"
]

for col in cols:
    df_accident_score[col] = pd.to_numeric(
        df_accident_score[col], errors="coerce"
    )

# Preliminary score based on reported road accident cases
min_cases = df_accident_score["Road Accidents - Cases"].min()
max_cases = df_accident_score["Road Accidents - Cases"].max()

if max_cases > min_cases:
    df_accident_score["accident_score"] = (
        100 * (max_cases - df_accident_score["Road Accidents - Cases"])
        / (max_cases - min_cases)
    ).round(2)
else:
    df_accident_score["accident_score"] = 100

display(df_accident_score.head(10))


,State/UT/City,Road Accidents - Cases,Road Accidents - Died,Road Accidents - Injured,accident_score
0,Andhra Pradesh,21070,8293,21340,95.28
1,Arunachal Pradesh,215,152,177,99.95
2,Assam,7028,3060,5679,98.43
3,Bihar,10801,8898,7068,97.58
4,Chhattisgarh,13091,5890,11459,97.07
5,Goa,3012,274,1071,99.33
6,Gujarat,15777,7634,15139,96.47
7,Haryana,10654,5228,8353,97.62
8,Himachal Pradesh,2484,979,3891,99.44
9,Jharkhand,5175,3898,3747,98.84


In [8]:

import os

os.makedirs("../data/processed", exist_ok=True)

df_accident_score.to_csv(
    "../data/processed/india_accident_scores_preliminary.csv",
    index=False
)

print("Preliminary accident scores saved successfully!")


Preliminary accident scores saved successfully!


In [9]:

import os
import shutil

os.makedirs("../data/final", exist_ok=True)

shutil.copy2(
    "../data/processed/india_accident_scores_preliminary.csv",
    "../data/final/india_accident_scores_preliminary.csv"
)

print("Preliminary accident scores copied to data/final!")


Preliminary accident scores copied to data/final!


In [1]:
import pandas as pd

file_path = "../data/final/india_accident_scores_preliminary.csv"
df_accidents = pd.read_csv(file_path)

print("Shape:", df_accidents.shape)
print("\nColumns:")
print(df_accidents.columns.tolist())

display(df_accidents.head())

Shape: (93, 5)

Columns:
['State/UT/City', 'Road Accidents - Cases', 'Road Accidents - Died', 'Road Accidents - Injured', 'accident_score']


,State/UT/City,Road Accidents - Cases,Road Accidents - Died,Road Accidents - Injured,accident_score
0,Andhra Pradesh,21070,8293,21340,95.28
1,Arunachal Pradesh,215,152,177,99.95
2,Assam,7028,3060,5679,98.43
3,Bihar,10801,8898,7068,97.58
4,Chhattisgarh,13091,5890,11459,97.07


In [2]:
# Check duplicate locations and missing values

print("Missing values:")
display(df_accidents.isnull().sum())

print("\nLocation names:")
display(df_accidents["State/UT/City"].tolist())

Missing values:


State/UT/City               0
Road Accidents - Cases      0
Road Accidents - Died       0
Road Accidents - Injured    0
accident_score              0
dtype: int64


Location names:


['Andhra Pradesh',
 'Arunachal Pradesh',
 'Assam',
 'Bihar',
 'Chhattisgarh',
 'Goa',
 'Gujarat',
 'Haryana',
 'Himachal Pradesh',
 'Jharkhand',
 'Karnataka',
 'Kerala',
 'Madhya Pradesh',
 'Maharashtra',
 'Manipur',
 'Meghalaya',
 'Mizoram',
 'Nagaland',
 'Odisha',
 'Punjab',
 'Rajasthan',
 'Sikkim',
 'Tamil Nadu',
 'Telangana',
 'Tripura',
 'Uttar Pradesh',
 'Uttarakhand',
 'West Bengal',
 'Total (States)',
 'Andaman and Nicobar Islands',
 'Chandigarh',
 'Dadra and Nicoabr Haveli and Daman and Diu',
 'Delhi',
 'Jammu and Kashmir',
 'Ladakh',
 'Lakshadweep',
 'Puducherry',
 'Total (UTs)',
 'Total (All India)',
 'Agra',
 'Ahmedabad',
 'Amritsar',
 'Asansol',
 'Aurangabad',
 'Bengaluru',
 'Bhopal',
 'Chandigarh (city)',
 'Chennai',
 'Coimbatore',
 'Delhi (city)',
 'Dhanbad',
 'Durg Bhilainagar',
 'Faridabad',
 'Ghaziabad',
 'Gwalior',
 'Hyderabad',
 'Indore',
 'Jabalpur',
 'Jaipur',
 'Jamshedpur',
 'Jodhpur',
 'Kannur',
 'Kanpur',
 'Kochi',
 'Kolkata',
 'Kollam',
 'Kota',
 'Kozhikode',


In [3]:
# Show only city-level entries, excluding states and summary rows

state_names = [
    "Andhra Pradesh", "Arunachal Pradesh", "Assam", "Bihar",
    "Chhattisgarh", "Goa", "Gujarat", "Haryana",
    "Himachal Pradesh", "Jharkhand", "Karnataka", "Kerala",
    "Madhya Pradesh", "Maharashtra", "Manipur", "Meghalaya",
    "Mizoram", "Nagaland", "Odisha", "Punjab", "Rajasthan",
    "Sikkim", "Tamil Nadu", "Telangana", "Tripura",
    "Uttar Pradesh", "Uttarakhand", "West Bengal",
    "Andaman and Nicobar Islands", "Chandigarh",
    "Dadra and Nagar Haveli and Daman and Diu",
    "Delhi", "Jammu and Kashmir", "Ladakh", "Lakshadweep",
    "Puducherry"
]

city_rows = df_accidents[
    ~df_accidents["State/UT/City"].isin(state_names)
    & ~df_accidents["State/UT/City"].str.startswith("Total", na=False)
]

print("City-level rows:", len(city_rows))
display(city_rows.head(15))

City-level rows: 54


,State/UT/City,Road Accidents - Cases,Road Accidents - Died,Road Accidents - Injured,accident_score
31,Dadra and Nicoabr Haveli and Daman and Diu,175,80,257,99.96
39,Agra,729,698,91,99.84
40,Ahmedabad,1711,488,1305,99.62
41,Amritsar,117,82,62,99.97
42,Asansol,497,365,308,99.89
43,Aurangabad,491,204,352,99.89
44,Bengaluru,3822,772,3189,99.15
45,Bhopal,2788,205,2151,99.38
46,Chandigarh (city),238,83,203,99.95
47,Chennai,3452,507,3724,99.23


In [4]:
import pandas as pd

df_city = city_rows.copy()

# Ensure numeric values
cols = [
    "Road Accidents - Cases",
    "Road Accidents - Died",
    "Road Accidents - Injured"
]

for col in cols:
    df_city[col] = pd.to_numeric(df_city[col], errors="coerce")

# Calculate fatality share among reported accidents
df_city["fatalities_per_100_accidents"] = (
    df_city["Road Accidents - Died"]
    / df_city["Road Accidents - Cases"].replace(0, pd.NA)
    * 100
).round(2)

display(
    df_city[
        [
            "State/UT/City",
            "Road Accidents - Cases",
            "Road Accidents - Died",
            "Road Accidents - Injured",
            "fatalities_per_100_accidents"
        ]
    ].head(15)
)

,State/UT/City,Road Accidents - Cases,Road Accidents - Died,Road Accidents - Injured,fatalities_per_100_accidents
31,Dadra and Nicoabr Haveli and Daman and Diu,175,80,257,45.71
39,Agra,729,698,91,95.75
40,Ahmedabad,1711,488,1305,28.52
41,Amritsar,117,82,62,70.09
42,Asansol,497,365,308,73.44
43,Aurangabad,491,204,352,41.55
44,Bengaluru,3822,772,3189,20.20
45,Bhopal,2788,205,2151,7.35
46,Chandigarh (city),238,83,203,34.87
47,Chennai,3452,507,3724,14.69


In [5]:
import os

# Keep only city-level rows for this output
df_city_accident_final = df_city[
    [
        "State/UT/City",
        "Road Accidents - Cases",
        "Road Accidents - Died",
        "Road Accidents - Injured",
        "fatalities_per_100_accidents"
    ]
].copy()

output_path = "../data/processed/india_city_accident_analysis.csv"

os.makedirs("../data/processed", exist_ok=True)
df_city_accident_final.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows:", len(df_city_accident_final))

Saved: ../data/processed/india_city_accident_analysis.csv
Rows: 54


In [1]:
import pandas as pd
from pathlib import Path

# Load the city-level accident data we already prepared
file_path = Path("../data/processed/india_city_accident_analysis.csv")
df = pd.read_csv(file_path)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

Shape: (54, 5)
Columns: ['State/UT/City', 'Road Accidents - Cases', 'Road Accidents - Died', 'Road Accidents - Injured', 'fatalities_per_100_accidents']


,State/UT/City,Road Accidents - Cases,Road Accidents - Died,Road Accidents - Injured,fatalities_per_100_accidents
0,Dadra and Nicoabr Haveli and Daman and Diu,175,80,257,45.71
1,Agra,729,698,91,95.75
2,Ahmedabad,1711,488,1305,28.52
3,Amritsar,117,82,62,70.09
4,Asansol,497,365,308,73.44


In [2]:

import pandas as pd
from pathlib import Path

# Load city-level accident data
df = pd.read_csv("../data/processed/india_city_accident_analysis.csv")

# Load the preliminary safety scores
df_scores = pd.read_csv("../data/final/india_accident_scores_preliminary.csv")

# Merge the score into the city-level data
df = df.merge(
    df_scores[["State/UT/City", "accident_score"]],
    on="State/UT/City",
    how="left"
)

# Convert to the backend team's requested format
backend_df = pd.DataFrame({
    "state": pd.NA,
    "city_district": df["State/UT/City"],
    "latitude": pd.NA,
    "longitude": pd.NA,
    "total_accidents": df["Road Accidents - Cases"],
    "total_deaths": df["Road Accidents - Died"],
    "total_injuries": df["Road Accidents - Injured"],
    "year": 2022,
    "accident_score": df["accident_score"]
})

# Save the review-ready CSV
output_path = Path("../data/final/accident_backend_review.csv")
backend_df.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Shape:", backend_df.shape)
display(backend_df.head(10))


Saved: ..\data\final\accident_backend_review.csv
Shape: (54, 9)


,state,city_district,latitude,longitude,total_accidents,total_deaths,total_injuries,year,accident_score
0,<NA>,Dadra and Nicoabr Haveli and Daman and Diu,<NA>,<NA>,175,80,257,2022,99.96
1,<NA>,Agra,<NA>,<NA>,729,698,91,2022,99.84
2,<NA>,Ahmedabad,<NA>,<NA>,1711,488,1305,2022,99.62
3,<NA>,Amritsar,<NA>,<NA>,117,82,62,2022,99.97
4,<NA>,Asansol,<NA>,<NA>,497,365,308,2022,99.89
5,<NA>,Aurangabad,<NA>,<NA>,491,204,352,2022,99.89
6,<NA>,Bengaluru,<NA>,<NA>,3822,772,3189,2022,99.15
7,<NA>,Bhopal,<NA>,<NA>,2788,205,2151,2022,99.38
8,<NA>,Chandigarh (city),<NA>,<NA>,238,83,203,2022,99.95
9,<NA>,Chennai,<NA>,<NA>,3452,507,3724,2022,99.23


In [3]:

print("Score range:")
print(backend_df["accident_score"].min(),
      backend_df["accident_score"].max())

print("\nLowest accident scores:")
display(
    backend_df[
        ["city_district", "total_accidents", "accident_score"]
    ].sort_values("accident_score").head(10)
)

print("\nMissing values:")
display(backend_df.isna().sum())


Score range:
98.8 99.97

Lowest accident scores:


,city_district,total_accidents,accident_score
11,Delhi (city),5387,98.80
6,Bengaluru,3822,99.15
9,Chennai,3452,99.23
18,Indore,3317,99.26
7,Bhopal,2788,99.38
20,Jaipur,2687,99.40
17,Hyderabad,2516,99.44
19,Jabalpur,2460,99.45
25,Kochi,2432,99.46
47,Thrissur,2374,99.47



Missing values:


state              54
city_district       0
latitude           54
longitude          54
total_accidents     0
total_deaths        0
total_injuries      0
year                0
accident_score      0
dtype: int64

In [4]:

# Recalculate a preliminary score using only these 54 location records
max_accidents = backend_df["total_accidents"].max()
min_accidents = backend_df["total_accidents"].min()

if max_accidents > min_accidents:
    backend_df["accident_score"] = (
        100 * (max_accidents - backend_df["total_accidents"])
        / (max_accidents - min_accidents)
    ).round(2)
else:
    backend_df["accident_score"] = 100

# Save the updated review CSV
backend_df.to_csv(
    "../data/final/accident_backend_review.csv",
    index=False
)

print("Updated score range:",
      backend_df["accident_score"].min(),
      "to",
      backend_df["accident_score"].max())

display(
    backend_df[
        ["city_district", "total_accidents", "accident_score"]
    ].sort_values("accident_score").head(10)
)


Updated score range: 0.0 to 100.0


,city_district,total_accidents,accident_score
11,Delhi (city),5387,0.00
6,Bengaluru,3822,29.70
9,Chennai,3452,36.72
18,Indore,3317,39.28
7,Bhopal,2788,49.32
20,Jaipur,2687,51.23
17,Hyderabad,2516,54.48
19,Jabalpur,2460,55.54
25,Kochi,2432,56.07
47,Thrissur,2374,57.17


In [5]:

import pandas as pd

df = pd.read_csv("../data/final/accident_backend_review.csv")

print("Total locations:", len(df))
print("Missing state:", df["state"].isna().sum())
print("Missing latitude:", df["latitude"].isna().sum())
print("Missing longitude:", df["longitude"].isna().sum())

print("\nAll location names:")
display(df[["city_district", "total_accidents", "year"]])


Total locations: 54
Missing state: 54
Missing latitude: 54
Missing longitude: 54

All location names:


,city_district,total_accidents,year
0,Dadra and Nicoabr Haveli and Daman and Diu,175,2022
1,Agra,729,2022
2,Ahmedabad,1711,2022
3,Amritsar,117,2022
4,Asansol,497,2022
5,Aurangabad,491,2022
6,Bengaluru,3822,2022
7,Bhopal,2788,2022
8,Chandigarh (city),238,2022
9,Chennai,3452,2022


In [6]:

import pandas as pd

df = pd.read_csv("../data/final/accident_backend_review.csv")

# Remove the state/UT row from the city-only review list
state_ut_row = "Dadra and Nicoabr Haveli and Daman and Diu"

city_df = df[df["city_district"] != state_ut_row].copy()

# Create a review file for verified state and coordinate mapping
city_df["state"] = ""
city_df["latitude"] = pd.NA
city_df["longitude"] = pd.NA

city_df.to_csv(
    "../data/processed/accident_city_location_mapping_review.csv",
    index=False
)

print("City records:", len(city_df))
print("Saved: data/processed/accident_city_location_mapping_review.csv")
display(city_df[["city_district", "state", "latitude", "longitude"]].head(10))


City records: 53
Saved: data/processed/accident_city_location_mapping_review.csv


,city_district,state,latitude,longitude
1,Agra,,<NA>,<NA>
2,Ahmedabad,,<NA>,<NA>
3,Amritsar,,<NA>,<NA>
4,Asansol,,<NA>,<NA>
5,Aurangabad,,<NA>,<NA>
6,Bengaluru,,<NA>,<NA>
7,Bhopal,,<NA>,<NA>
8,Chandigarh (city),,<NA>,<NA>
9,Chennai,,<NA>,<NA>
10,Coimbatore,,<NA>,<NA>


In [7]:

import pandas as pd

path = "../data/processed/accident_city_location_mapping_review.csv"
df = pd.read_csv(path)

city_to_state = {
    "Agra": "Uttar Pradesh",
    "Ahmedabad": "Gujarat",
    "Amritsar": "Punjab",
    "Asansol": "West Bengal",
    "Aurangabad": "Maharashtra",
    "Bengaluru": "Karnataka",
    "Bhopal": "Madhya Pradesh",
    "Chandigarh (city)": "Chandigarh",
    "Chennai": "Tamil Nadu",
    "Coimbatore": "Tamil Nadu",
    "Delhi (city)": "Delhi",
    "Dhanbad": "Jharkhand",
    "Durg Bhilainagar": "Chhattisgarh",
    "Faridabad": "Haryana",
    "Ghaziabad": "Uttar Pradesh",
    "Gwalior": "Madhya Pradesh",
    "Hyderabad": "Telangana",
    "Indore": "Madhya Pradesh",
    "Jabalpur": "Madhya Pradesh",
    "Jaipur": "Rajasthan",
    "Jamshedpur": "Jharkhand",
    "Jodhpur": "Rajasthan",
    "Kannur": "Kerala",
    "Kanpur": "Uttar Pradesh",
    "Kochi": "Kerala",
    "Kolkata": "West Bengal",
    "Kollam": "Kerala",
    "Kota": "Rajasthan",
    "Kozhikode": "Kerala",
    "Lucknow": "Uttar Pradesh",
    "Ludhiana": "Punjab",
    "Madurai": "Tamil Nadu",
    "Malappuram": "Kerala",
    "Meerut": "Uttar Pradesh",
    "Mumbai": "Maharashtra",
    "Nagpur": "Maharashtra",
    "Nasik": "Maharashtra",
    "Patna": "Bihar",
    "Prayagraj +": "Uttar Pradesh",
    "Pune": "Maharashtra",
    "Raipur": "Chhattisgarh",
    "Rajkot": "Gujarat",
    "Ranchi": "Jharkhand",
    "Srinagar": "Jammu and Kashmir",
    "Surat": "Gujarat",
    "Thiruvananthapuram": "Kerala",
    "Thrissur": "Kerala",
    "Tiruchirappalli": "Tamil Nadu",
    "Vadodara": "Gujarat",
    "Varanasi": "Uttar Pradesh",
    "Vasai Virar": "Maharashtra",
    "Vijayawada": "Andhra Pradesh",
    "Vishakhapatnam": "Andhra Pradesh"
}

df["state"] = df["city_district"].map(city_to_state)

df.to_csv(path, index=False)

print("Total records:", len(df))
print("State names missing:", df["state"].isna().sum())
display(df[["city_district", "state"]])


Total records: 53
State names missing: 0


,city_district,state
0,Agra,Uttar Pradesh
1,Ahmedabad,Gujarat
2,Amritsar,Punjab
3,Asansol,West Bengal
4,Aurangabad,Maharashtra
5,Bengaluru,Karnataka
6,Bhopal,Madhya Pradesh
7,Chandigarh (city),Chandigarh
8,Chennai,Tamil Nadu
9,Coimbatore,Tamil Nadu


In [8]:

import pandas as pd

df = pd.read_csv(
    "../data/processed/accident_city_location_mapping_review.csv"
)

# Mark ambiguous names for manual verification
ambiguous = [
    "Aurangabad",
    "Chandigarh (city)",
    "Delhi (city)",
    "Durg Bhilainagar",
    "Nasik",
    "Prayagraj +",
    "Vasai Virar",
    "Vishakhapatnam"
]

df["location_needs_review"] = df["city_district"].isin(ambiguous)

print("Total city records:", len(df))
print("Names needing review:", df["location_needs_review"].sum())

display(
    df.loc[
        df["location_needs_review"],
        ["city_district", "state", "location_needs_review"]
    ]
)


Total city records: 53
Names needing review: 8


,city_district,state,location_needs_review
4,Aurangabad,Maharashtra,True
7,Chandigarh (city),Chandigarh,True
10,Delhi (city),Delhi,True
12,Durg Bhilainagar,Chhattisgarh,True
36,Nasik,Maharashtra,True
38,Prayagraj +,Uttar Pradesh,True
50,Vasai Virar,Maharashtra,True
52,Vishakhapatnam,Andhra Pradesh,True


In [10]:

import pandas as pd

# Load the city location mapping review file
file_path = "../data/processed/accident_city_location_mapping_review.csv"
df_mapping = pd.read_csv(file_path)

# Display the total number of city records
print("Total city records:", len(df_mapping))

# Display all available columns
print("\nAvailable columns:")
print(df_mapping.columns.tolist())

# Identify cities with missing coordinates
missing_coordinates = df_mapping[
    df_mapping["latitude"].isna() |
    df_mapping["longitude"].isna()
]

# Display cities with missing coordinates
print("\nCities with missing coordinates:")
display(
    missing_coordinates[
        ["state", "city_district", "latitude", "longitude"]
    ]
)


Total city records: 53

Available columns:
['state', 'city_district', 'latitude', 'longitude', 'total_accidents', 'total_deaths', 'total_injuries', 'year', 'accident_score']

Cities with missing coordinates:


,state,city_district,latitude,longitude
0,Uttar Pradesh,Agra,NaN,NaN
1,Gujarat,Ahmedabad,NaN,NaN
2,Punjab,Amritsar,NaN,NaN
3,West Bengal,Asansol,NaN,NaN
4,Maharashtra,Aurangabad,NaN,NaN
5,Karnataka,Bengaluru,NaN,NaN
6,Madhya Pradesh,Bhopal,NaN,NaN
7,Chandigarh,Chandigarh (city),NaN,NaN
8,Tamil Nadu,Chennai,NaN,NaN
9,Tamil Nadu,Coimbatore,NaN,NaN


In [11]:

import pandas as pd

# Load the city mapping review file
file_path = "../data/processed/accident_city_location_mapping_review.csv"
df_mapping = pd.read_csv(file_path)

# Reference coordinates for city centers
city_coordinates = {
    "Agra": (27.1767, 78.0081),
    "Ahmedabad": (23.0225, 72.5714),
    "Amritsar": (31.6340, 74.8723),
    "Asansol": (23.6739, 86.9524),
    "Aurangabad": (19.8762, 75.3433),
    "Bengaluru": (12.9716, 77.5946),
    "Bhopal": (23.2599, 77.4126),
    "Chandigarh (city)": (30.7333, 76.7794),
    "Chennai": (13.0827, 80.2707),
    "Coimbatore": (11.0168, 76.9558),
    "Delhi (city)": (28.6139, 77.2090),
    "Dhanbad": (23.7957, 86.4304),
    "Faridabad": (28.4089, 77.3178),
    "Ghaziabad": (28.6692, 77.4538),
    "Gwalior": (26.2183, 78.1828),
    "Hyderabad": (17.3850, 78.4867),
    "Indore": (22.7196, 75.8577),
    "Jabalpur": (23.1815, 79.9864),
    "Jaipur": (26.9124, 75.7873),
    "Jamshedpur": (22.8046, 86.2029),
    "Jodhpur": (26.2389, 73.0243),
    "Kannur": (11.8745, 75.3704),
    "Kanpur": (26.4499, 80.3319),
    "Kochi": (9.9312, 76.2673),
    "Kolkata": (22.5726, 88.3639),
    "Kollam": (8.8932, 76.6141),
    "Kota": (25.2138, 75.8648),
    "Kozhikode": (11.2588, 75.7804),
    "Lucknow": (26.8467, 80.9462),
    "Ludhiana": (30.9010, 75.8573),
    "Madurai": (9.9252, 78.1198),
    "Malappuram": (11.0510, 76.0711),
    "Meerut": (28.9845, 77.7064),
    "Mumbai": (19.0760, 72.8777),
    "Nagpur": (21.1458, 79.0882),
    "Nasik": (19.9975, 73.7898),
    "Patna": (25.5941, 85.1376),
    "Pune": (18.5204, 73.8567),
    "Raipur": (21.2514, 81.6296),
    "Rajkot": (22.3039, 70.8022),
    "Ranchi": (23.3441, 85.3096),
    "Srinagar": (34.0837, 74.7973),
    "Surat": (21.1702, 72.8311),
    "Thiruvananthapuram": (8.5241, 76.9366),
    "Thrissur": (10.5276, 76.2144),
    "Tiruchirappalli": (10.7905, 78.7047),
    "Vadodara": (22.3072, 73.1812),
    "Varanasi": (25.3176, 82.9739),
    "Vijayawada": (16.5062, 80.6480),
    "Vishakhapatnam": (17.6868, 83.2185)
}

# Apply coordinates to matching city names
for index, row in df_mapping.iterrows():
    city = row["city_district"]

    if city in city_coordinates:
        df_mapping.at[index, "latitude"] = city_coordinates[city][0]
        df_mapping.at[index, "longitude"] = city_coordinates[city][1]

# Save the updated mapping review file
df_mapping.to_csv(file_path, index=False)

# Display the mapping results
missing = df_mapping[
    df_mapping["latitude"].isna() |
    df_mapping["longitude"].isna()
]

print("Total city records:", len(df_mapping))
print("Cities with coordinates:", len(df_mapping) - len(missing))
print("Cities still missing coordinates:", len(missing))

display(
    df_mapping[
        ["state", "city_district", "latitude", "longitude"]
    ]
)


Total city records: 53
Cities with coordinates: 50
Cities still missing coordinates: 3


,state,city_district,latitude,longitude
0,Uttar Pradesh,Agra,27.1767,78.0081
1,Gujarat,Ahmedabad,23.0225,72.5714
2,Punjab,Amritsar,31.6340,74.8723
3,West Bengal,Asansol,23.6739,86.9524
4,Maharashtra,Aurangabad,19.8762,75.3433
5,Karnataka,Bengaluru,12.9716,77.5946
6,Madhya Pradesh,Bhopal,23.2599,77.4126
7,Chandigarh,Chandigarh (city),30.7333,76.7794
8,Tamil Nadu,Chennai,13.0827,80.2707
9,Tamil Nadu,Coimbatore,11.0168,76.9558


In [1]:

import pandas as pd

file_path = "../data/processed/accident_city_location_mapping_review.csv"
df_mapping = pd.read_csv(file_path)

# Normalize the Prayagraj city name
df_mapping["city_district"] = (
    df_mapping["city_district"]
    .astype(str)
    .str.strip()
    .replace({"Prayagraj +": "Prayagraj"})
)

# Set the official district-centre coordinates for Prayagraj
prayagraj_mask = df_mapping["city_district"] == "Prayagraj"

df_mapping.loc[prayagraj_mask, "latitude"] = 25.45
df_mapping.loc[prayagraj_mask, "longitude"] = 81.84

# Save the updated mapping review file
df_mapping.to_csv(file_path, index=False)

# Review any remaining missing coordinates
missing_coordinates = df_mapping[
    df_mapping["latitude"].isna() |
    df_mapping["longitude"].isna()
]

print("Total city records:", len(df_mapping))
print("Cities with coordinates:", len(df_mapping) - len(missing_coordinates))
print("Cities still missing coordinates:", len(missing_coordinates))

display(
    missing_coordinates[
        ["state", "city_district", "latitude", "longitude"]
    ]
)


Total city records: 53
Cities with coordinates: 51
Cities still missing coordinates: 2


,state,city_district,latitude,longitude
12,Chhattisgarh,Durg Bhilainagar,NaN,NaN
50,Maharashtra,Vasai Virar,NaN,NaN


In [2]:

import pandas as pd

file_path = "../data/processed/accident_city_location_mapping_review.csv"
df_mapping = pd.read_csv(file_path)

# Set approximate coordinates for the Durg-Bhilainagar urban cluster
durg_mask = df_mapping["city_district"] == "Durg Bhilainagar"
df_mapping.loc[durg_mask, "latitude"] = 21.20
df_mapping.loc[durg_mask, "longitude"] = 81.32

# Set the Vasai-Virar municipal corporation office coordinates
vasai_mask = df_mapping["city_district"] == "Vasai Virar"
df_mapping.loc[vasai_mask, "latitude"] = 19.45611
df_mapping.loc[vasai_mask, "longitude"] = 72.81253

# Save the updated mapping review file
df_mapping.to_csv(file_path, index=False)

# Check coordinate completeness
missing_coordinates = df_mapping[
    df_mapping["latitude"].isna() |
    df_mapping["longitude"].isna()
]

print("Total city records:", len(df_mapping))
print("Cities with coordinates:", len(df_mapping) - len(missing_coordinates))
print("Cities still missing coordinates:", len(missing_coordinates))

display(
    df_mapping[
        ["state", "city_district", "latitude", "longitude"]
    ]
)


Total city records: 53
Cities with coordinates: 53
Cities still missing coordinates: 0


,state,city_district,latitude,longitude
0,Uttar Pradesh,Agra,27.17670,78.00810
1,Gujarat,Ahmedabad,23.02250,72.57140
2,Punjab,Amritsar,31.63400,74.87230
3,West Bengal,Asansol,23.67390,86.95240
4,Maharashtra,Aurangabad,19.87620,75.34330
5,Karnataka,Bengaluru,12.97160,77.59460
6,Madhya Pradesh,Bhopal,23.25990,77.41260
7,Chandigarh,Chandigarh (city),30.73330,76.77940
8,Tamil Nadu,Chennai,13.08270,80.27070
9,Tamil Nadu,Coimbatore,11.01680,76.95580


In [4]:

import pandas as pd
from pathlib import Path

# Load the completed city coordinate mapping
input_path = "../data/processed/accident_city_location_mapping_review.csv"
df_accident = pd.read_csv(input_path)

# Define the required columns
required_columns = [
    "state",
    "city_district",
    "latitude",
    "longitude",
    "total_accidents",
    "total_deaths",
    "total_injuries",
    "year",
    "accident_score"
]

# Check whether all required columns exist
missing_columns = [
    column for column in required_columns
    if column not in df_accident.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# Check for missing coordinates
if df_accident[["latitude", "longitude"]].isna().any().any():
    raise ValueError("Some cities still have missing coordinates.")

# Select the final columns
df_final = df_accident[required_columns].copy()

# Add source and interpretation notes
df_final["data_source"] = "NCRB ADSI Table 1A.2, 2022"
df_final["coordinate_type"] = "Approximate city-centre coordinates"
df_final["score_note"] = (
    "Preliminary inverse min-max score based on reported accident counts "
    "across the 53 listed cities; not exposure-adjusted"
)

# Save the final CSV
output_path = Path("../data/final/accident_city_safety_scores.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
df_final.to_csv(output_path, index=False)

# Verify the saved file
df_check = pd.read_csv(output_path)

print("Final file saved:", output_path)
print("Total records:", len(df_check))
print(
    "Missing coordinates:",
    df_check[["latitude", "longitude"]].isna().sum().sum()
)
print("\nMissing values by column:")
print(df_check.isna().sum())

display(df_check.head())


Final file saved: ..\data\final\accident_city_safety_scores.csv
Total records: 53
Missing coordinates: 0

Missing values by column:
state              0
city_district      0
latitude           0
longitude          0
total_accidents    0
total_deaths       0
total_injuries     0
year               0
accident_score     0
data_source        0
coordinate_type    0
score_note         0
dtype: int64


,state,city_district,latitude,longitude,total_accidents,total_deaths,total_injuries,year,accident_score,data_source,coordinate_type,score_note
0,Uttar Pradesh,Agra,27.1767,78.0081,729,698,91,2022,88.39,"NCRB ADSI Table 1A.2, 2022",Approximate city-centre coordinates,Preliminary inverse min-max score based on rep...
1,Gujarat,Ahmedabad,23.0225,72.5714,1711,488,1305,2022,69.75,"NCRB ADSI Table 1A.2, 2022",Approximate city-centre coordinates,Preliminary inverse min-max score based on rep...
2,Punjab,Amritsar,31.6340,74.8723,117,82,62,2022,100.00,"NCRB ADSI Table 1A.2, 2022",Approximate city-centre coordinates,Preliminary inverse min-max score based on rep...
3,West Bengal,Asansol,23.6739,86.9524,497,365,308,2022,92.79,"NCRB ADSI Table 1A.2, 2022",Approximate city-centre coordinates,Preliminary inverse min-max score based on rep...
4,Maharashtra,Aurangabad,19.8762,75.3433,491,204,352,2022,92.90,"NCRB ADSI Table 1A.2, 2022",Approximate city-centre coordinates,Preliminary inverse min-max score based on rep...


In [5]:

import pandas as pd

# Load the final accident safety dataset
file_path = "../data/final/accident_city_safety_scores.csv"
df_check = pd.read_csv(file_path)

# Display the accident score range
print("Minimum accident score:", df_check["accident_score"].min())
print("Maximum accident score:", df_check["accident_score"].max())

# Display cities with the highest and lowest accident scores
print("\nFive cities with the highest scores:")
display(
    df_check.nlargest(5, "accident_score")[
        ["state", "city_district", "total_accidents", "accident_score"]
    ]
)

print("\nFive cities with the lowest scores:")
display(
    df_check.nsmallest(5, "accident_score")[
        ["state", "city_district", "total_accidents", "accident_score"]
    ]
)

# Check for invalid score values
invalid_scores = df_check[
    (df_check["accident_score"] < 0) |
    (df_check["accident_score"] > 100)
]

print("\nScores outside the 0-100 range:", len(invalid_scores))


Minimum accident score: 0.0
Maximum accident score: 100.0

Five cities with the highest scores:


,state,city_district,total_accidents,accident_score
2,Punjab,Amritsar,117,100.00
49,Uttar Pradesh,Varanasi,133,99.70
20,Jharkhand,Jamshedpur,167,99.05
42,Jharkhand,Ranchi,175,98.90
11,Jharkhand,Dhanbad,179,98.82



Five cities with the lowest scores:


,state,city_district,total_accidents,accident_score
10,Delhi,Delhi (city),5387,0.00
5,Karnataka,Bengaluru,3822,29.70
8,Tamil Nadu,Chennai,3452,36.72
17,Madhya Pradesh,Indore,3317,39.28
6,Madhya Pradesh,Bhopal,2788,49.32



Scores outside the 0-100 range: 0


In [6]:

import pandas as pd
from pathlib import Path

# Define the final dataset files
data_files = {
    "Crime": "../data/final/crime_safety_scores.csv",
    "Traffic": "../data/final/traffic_pune_sample.csv",
    "Road": "../data/final/road_condition_scores.csv",
    "Accident": "../data/final/accident_city_safety_scores.csv"
}

# Check each dataset
for dataset_name, file_path in data_files.items():
    path = Path(file_path)

    print("\n" + "=" * 50)
    print("Dataset:", dataset_name)
    print("File exists:", path.exists())

    if path.exists():
        df = pd.read_csv(path)

        print("Rows:", len(df))
        print("Columns:", df.columns.tolist())
        print("Missing values:", int(df.isna().sum().sum()))

        coordinate_columns = {"latitude", "longitude"}
        if coordinate_columns.issubset(df.columns):
            missing_coordinates = df[
                df[["latitude", "longitude"]].isna().any(axis=1)
            ]
            print("Rows with missing coordinates:", len(missing_coordinates))
        else:
            print("Coordinate columns: Not found")

        score_columns = [
            column for column in df.columns
            if "score" in column.lower()
        ]
        print("Score columns:", score_columns)

    else:
        print("Please check the file name and folder.")



Dataset: Crime
File exists: True
Rows: 480
Columns: ['STATE/UT', 'DISTRICT', 'latitude', 'longitude', 'total_crimes', 'crime_score', 'coordinate_source']
Missing values: 0
Rows with missing coordinates: 0
Score columns: ['crime_score']

Dataset: Traffic
File exists: True
Rows: 5
Columns: ['latitude', 'longitude', 'current_speed_kmph', 'free_flow_speed_kmph', 'traffic_score', 'data_source', 'congestion_percent', 'collected_at_utc']
Missing values: 0
Rows with missing coordinates: 0
Score columns: ['traffic_score']

Dataset: Road
File exists: True
Rows: 9
Columns: ['id', 'lat', 'lng', 'road_type', 'road_name', 'severity', 'status', 'state', 'district', 'upvotes', 'photo_count', 'first_reported', 'last_reported', 'url', 'road_score']
Missing values: 27
Coordinate columns: Not found
Score columns: ['road_score']

Dataset: Accident
File exists: True
Rows: 53
Columns: ['state', 'city_district', 'latitude', 'longitude', 'total_accidents', 'total_deaths', 'total_injuries', 'year', 'accident_s

In [7]:

import pandas as pd

# Load the road condition dataset
file_path = "../data/final/road_condition_scores.csv"
df_road = pd.read_csv(file_path)

# Count missing values in each column
print("Missing values by column:")
print(df_road.isna().sum())

# Display records with missing values
print("\nRecords with missing values:")
display(df_road[df_road.isna().any(axis=1)])

# Check whether coordinates and scores are available
print("\nMissing latitude values:", df_road["lat"].isna().sum())
print("Missing longitude values:", df_road["lng"].isna().sum())
print("Missing road scores:", df_road["road_score"].isna().sum())

# Display the available road score range
print("Minimum road score:", df_road["road_score"].min())
print("Maximum road score:", df_road["road_score"].max())


Missing values by column:
id                0
lat               0
lng               0
road_type         0
road_name         9
severity          0
status            0
state             9
district          9
upvotes           0
photo_count       0
first_reported    0
last_reported     0
url               0
road_score        0
dtype: int64

Records with missing values:


,id,lat,lng,road_type,road_name,severity,status,state,district,upvotes,photo_count,first_reported,last_reported,url,road_score
0,1,12.987959,77.697255,city_road,NaN,2,open,NaN,NaN,8,2,2026-09-27T13:17:49.000Z,2026-09-27T13:18:30.000Z,https://kamartod.in/pothole.html?id=1,33.33
1,2,12.944594,77.680272,city_road,NaN,2,open,NaN,NaN,1,1,2026-09-30T12:08:45.000Z,2026-09-30T12:08:45.000Z,https://kamartod.in/pothole.html?id=2,33.33
2,3,12.947471,77.680687,city_road,NaN,3,open,NaN,NaN,1,1,2026-09-30T12:10:06.000Z,2026-09-30T12:10:06.000Z,https://kamartod.in/pothole.html?id=3,0.00
3,4,12.947533,77.680691,city_road,NaN,2,open,NaN,NaN,1,1,2026-09-30T12:10:38.000Z,2026-09-30T12:10:38.000Z,https://kamartod.in/pothole.html?id=4,33.33
4,5,12.948725,77.684794,city_road,NaN,1,open,NaN,NaN,1,1,2026-09-30T12:11:01.000Z,2026-09-30T12:11:01.000Z,https://kamartod.in/pothole.html?id=5,66.67
5,6,12.954136,77.689282,city_road,NaN,3,open,NaN,NaN,1,1,2026-09-30T12:17:46.000Z,2026-09-30T12:17:46.000Z,https://kamartod.in/pothole.html?id=6,0.00
6,7,12.972960,77.693584,local_street,NaN,2,open,NaN,NaN,1,1,2026-09-30T12:38:22.000Z,2026-09-30T12:38:22.000Z,https://kamartod.in/pothole.html?id=7,33.33
7,8,12.972989,77.694038,local_street,NaN,3,open,NaN,NaN,1,1,2026-09-30T12:41:34.000Z,2026-09-30T12:41:34.000Z,https://kamartod.in/pothole.html?id=8,0.00
8,9,12.980571,77.694989,city_road,NaN,2,open,NaN,NaN,1,1,2026-09-30T13:00:27.000Z,2026-09-30T13:00:27.000Z,https://kamartod.in/pothole.html?id=9,33.33



Missing latitude values: 0
Missing longitude values: 0
Missing road scores: 0
Minimum road score: 0.0
Maximum road score: 66.67
